# Lecture 12: Singular arcs: Goddard's rocket

**The challenge.** A sounding rocket climbs vertically through the
atmosphere on a fixed load of propellant; the goal is the highest altitude.
Thrust is bounded, $0 \le u \le u_{\max}$, and enters the dynamics linearly,
so Lecture 4 says: full thrust or none, decided by the sign of a switching
function. Full thrust until the propellant runs out, then coast? That throws
away energy fighting drag in the dense low air, where speed is most
expensive. The best climb does something the bang-bang law cannot express:
it throttles back to an *intermediate* thrust for a while. On that stretch
the switching function is not positive or negative. It is identically zero,
and the minimum principle, which only reads its sign, is silent.

Such a stretch is a **singular arc**. This lecture shows how to find the
control on it, how to check it is a minimum, how to join it to the
bang-bang arcs around it, and how the package solves for the whole
structure.

**After this lecture you can**

- find the singular control by differentiating the switching function, and
  define the order of a singular arc;
- apply the generalized Legendre–Clebsch condition and the junction
  conditions;
- use `derive(..., singular=True)`, `MultiArcShooting` with singular arcs and
  switches, and its diagnostics.

## 1. The rocket man

Robert Hutchings Goddard (1882–1945), a physics professor at Clark
University in Worcester, Massachusetts, published *A Method of Reaching
Extreme Altitudes* with the Smithsonian Institution in 1919. In it he worked
out, among much else, how a rocket's thrust should be programmed to climb
highest against drag: the problem that still carries his name. He also
remarked that a large enough rocket could reach the Moon, and suggested that
a charge of flash powder would show it had arrived.

On 13 January 1920 *The New York Times* answered in an editorial. A rocket
could not work in the vacuum of space, it said, since it would have nothing
to push against; Professor Goddard "seems to lack the knowledge ladled out
daily in high schools". Goddard, a private man, was stung, and he worked in
near-secrecy from then on. On 16 March 1926, on his aunt's farm in Auburn,
Massachusetts, he launched the first liquid-fuelled rocket. It flew for two
and a half seconds and landed 56 metres away. He died in 1945, before his
work was widely recognized. On 17 July 1969, the day after Apollo 11 left
for the Moon, the *Times* printed a correction: "it is now definitely
established that a rocket can function in a vacuum as well as in an
atmosphere. The Times regrets the error."

The optimal thrust program was found by Hsue-shen Tsien (Qian Xuesen) and
Robert Evans at Caltech in 1951. It includes an arc of intermediate thrust
on which drag and climb are balanced. Tsien, a co-founder of the Jet
Propulsion Laboratory, lost his security clearance in 1950, in the
McCarthy-era suspicion of anyone with ties to Communist China. He was held
under house arrest for five years and deported in 1955 to China, where he led
its missile and space programmes. The general theory of such arcs came in the
1960s: Henry Kelley at Grumman found the second-order condition they must
satisfy (1964), generalized by Kelley, Kopp and Moyer in 1967, and A. T.
Fuller in Cambridge showed in 1960 that some singular arcs cannot be joined
to bang-bang arcs at all without infinitely many switches.

## 2. Finding the singular control

On a singular arc $S \equiv 0$ over an interval, so all its time derivatives
vanish there too. Differentiate $S$ along the state–costate dynamics, with
the control left free, until the control appears:
$$
S = 0, \qquad \dot S = 0, \qquad \ddot S = a(x, \lambda) + b(x, \lambda)\,u = 0
\;\Rightarrow\; u_s = -\frac{a}{b} .
$$
The control always appears first in an *even* derivative, $S^{(2q)}$; $q$
is the **order** of the arc. Two more conditions complete the picture:

- **Generalized Legendre–Clebsch** (Kelley, Kopp and Moyer):
  $(-1)^q\,\partial S^{(2q)}/\partial u \ge 0$ along a minimizing arc. For
  order 1 that is $-b \ge 0$. It plays the role of Legendre's $H_{uu} \ge 0$
  for a Hamiltonian that is linear in $u$.
- **Junctions.** Entering an order-1 arc from a bang-bang arc needs
  $S = \dot S = 0$ at the entry time; the singular control then keeps them at
  zero. $H$ is continuous at both junctions because $S = 0$ there, and the
  control may jump. For order 2, Fuller's result says a bang-bang arc cannot
  join the singular arc directly: the switches pile up infinitely fast
  (**chattering**). The package supports order 1 and reports the rest.

### The simplest singular arc, by hand

Take $\min\int x^2$ with $\dot x = u$, $|u| \le 1$, from $x(0) = a$ back to
$x(T) = a$. $H = x^2 + \lambda u$, so $S = \lambda$,
$\dot S = \dot\lambda = -2x$, and $\ddot S = -2\dot x = -2u$. So $u_s = 0$ on
the arc, which is $x = 0$ (from $\dot S = 0$), and the Legendre–Clebsch
coefficient is $-\partial\ddot S/\partial u = 2 > 0$. The solution is clear
once seen: drive to the origin at full speed, sit there, drive back.
`derive(problem, singular=True)` does the differentiation for every
bang-bang control:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from auto_optimal_control import examples
from auto_optimal_control.build import build_kernels
from auto_optimal_control.pontryagin import derive
from auto_optimal_control.shooting import MultiArcShooting

BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#8a8a85"

sr = derive(examples.singular_regulator(), singular=True)
arc = sr.singular_arcs[0]
print("S, S', S'' =", arc.derivatives)
print("singular control:", arc.law, "  Legendre-Clebsch:", arc.legendre_clebsch)
print("junction conditions (S, S') =", arc.junction)
print("minimum-time double integrator:",
      derive(examples.double_integrator_min_time(), singular=True).singular_unsupported[0])

The double integrator's switching function cannot vanish on an interval at
all ($\ddot S = 0$ identically, with no $u$), and `derive` says why there is
no singular arc to generate.

## 3. Goddard's problem

In normalized units (Seywald and Cliff, 1992), altitude $h$ from the Earth's
centre, speed $v$, mass $m$:
$$
\dot h = v, \qquad \dot v = \frac{u - D(h, v)}{m} - \frac{1}{h^2}, \qquad \dot m = -\frac{u}{c},
\qquad D = D_0 v^2 e^{-\beta(h - 1)},
$$
from rest at $h = 1$, $m = 1$, burning down to $m(t_f) = 0.6$, maximizing
$h(t_f)$ with $t_f$ free. With $D_0 = 310$, $\beta = 500$, $c = 0.5$,
$u_{\max} = 3.5$, the reference solution (Graichen and Petit, 2008) has
$h(t_f) = 1.01283692$.

In [ ]:
# Goddard's problem: a rocket climbing vertically through an atmosphere that thins
# exponentially. Thrust u (at most u_max) burns mass; drag grows with v^2 and air density.
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Polygon
import itertools

BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#8a8a85"
fig, ax = plt.subplots(figsize=(7.5, 5))
heights = np.linspace(0, 1, 200)
for y0, y1 in itertools.pairwise(heights):
    ax.axhspan(y0, y1, xmin=0, xmax=1, color=BLUE, alpha=0.35 * np.exp(-4 * y0), lw=0)
ax.text(0.03, 0.04, "dense air", fontsize=9)
ax.text(0.03, 0.85, r"thin air: density $\propto e^{-\beta (h - 1)}$", fontsize=9)
ax.fill_between([0, 1], -0.08, 0, color=GRAY, alpha=0.6, lw=0)
ax.text(0.5, -0.05, "Earth's surface, h = 1, start at rest", ha="center", va="center", fontsize=9)
cx, cy = 0.5, 0.48
body = Polygon([[cx - 0.035, cy - 0.12], [cx + 0.035, cy - 0.12], [cx + 0.035, cy + 0.08],
                [cx, cy + 0.15], [cx - 0.035, cy + 0.08]], closed=True, fc="white", ec="k", lw=1.5)
ax.add_patch(body)
ax.add_patch(Polygon([[cx - 0.03, cy - 0.12], [cx + 0.03, cy - 0.12], [cx, cy - 0.22]],
                     closed=True, fc=ORANGE, ec="none", alpha=0.8))
ax.text(cx, cy - 0.02, "m", ha="center", fontsize=11, style="italic")
arrow = {"arrowstyle": "-|>", "lw": 2.2, "mutation_scale": 16}
ax.annotate("", xy=(cx + 0.12, cy + 0.17), xytext=(cx + 0.12, cy - 0.03), arrowprops=dict(color=ORANGE, **arrow))
ax.text(cx + 0.14, cy + 0.08, r"thrust $u \in [0, u_{\max}]$" "\n" r"burns mass: $\dot m = -u/c$",
        fontsize=9, color=ORANGE, va="center")
ax.annotate("", xy=(cx - 0.12, cy - 0.12), xytext=(cx - 0.12, cy + 0.06), arrowprops=dict(color=BLUE, **arrow))
ax.text(cx - 0.14, cy - 0.02, r"drag $D = D_0 v^2 e^{-\beta (h - 1)}$", fontsize=9, color=BLUE,
        ha="right", va="center")
ax.annotate("", xy=(cx - 0.12, cy - 0.3), xytext=(cx - 0.12, cy - 0.16), arrowprops=dict(color="k", **arrow))
ax.text(cx - 0.14, cy - 0.23, r"gravity $m / h^2$", fontsize=9, ha="right", va="center")
ax.annotate("", xy=(cx, 0.0), xytext=(cx, cy - 0.23), arrowprops={"arrowstyle": "<->", "lw": 0.8, "color": GRAY})
ax.text(cx + 0.015, 0.12, "height h - 1 above the surface;\nspeed v upwards", fontsize=9, color="k")
ax.text(0.5, 0.97, r"maximize the final altitude $h(t_f)$ with the propellant burned, $m(t_f) = 0.6$",
        ha="center", va="top", fontsize=10)
ax.set(xlim=(0, 1), ylim=(-0.08, 1))
ax.axis("off")
fig.tight_layout()

*Goddard's problem. Full thrust low down wastes propellant fighting drag in dense air. No thrust wastes the climb. The answer is in between, on a singular arc.*

In [ ]:
goddard = examples.goddard()
pg = derive(goddard, singular=True)
ga = pg.singular_arcs[0]
print("S =", ga.derivatives[0])
print("u_s has", len(str(ga.law[pg.controls[0]])), "characters; Legendre-Clebsch has",
      len(str(ga.legendre_clebsch)))
kg = build_kernels(pg)
P = list(examples.GODDARD_PARAMS)
print("singular arc kernels generated:", kg.SINGULAR_ARCS)

First, the naive program: full thrust until the propellant is gone, then
coast to the top.

In [ ]:
from scipy.integrate import solve_ivp

D0, beta, c, umax, mf = P


def climb(t, y, thrust):
    h, v, m = y
    return [v, (thrust - D0 * v**2 * np.exp(-beta * (h - 1))) / m - 1 / h**2, -thrust / c]


t_burn = (1 - mf) * c / umax
burn = solve_ivp(climb, (0, t_burn), [1, 0, 1], args=(umax,), rtol=1e-12, atol=1e-12)
top = lambda t, y, thrust: y[1]
top.terminal = True
coast = solve_ivp(climb, (t_burn, 1.0), burn.y[:, -1], args=(0.0,), events=top,
                  rtol=1e-12, atol=1e-12)
h_naive = coast.y[0, -1]
print(f"full thrust then coast: h_max = {h_naive:.8f}")

### Solving with the structure given

`MultiArcShooting` (Lecture 14 covers it for constrained arcs) solves the
boundary value problem for a *given* sequence of special arcs. For Goddard
the structure is: full thrust, a singular arc, then no thrust:
`("singular", "u", "upper", "lower")`, which names the bound held on the
free arcs before and after. A control with a singular arc *holds* its bound
on the free arcs, and its switching function is not watched there, because at
a singular junction it touches zero with $\dot S = 0$, exactly the tangential
case Lecture 7 cannot locate. If it switches elsewhere, those switches are
listed as `("switch", u)` entries.

The unknowns are those of single shooting plus the entry and exit times.
The residuals add $S = \dot S = 0$ at the entry.

In [ ]:
shoot = MultiArcShooting(kg, P, [("singular", "u", "upper", "lower")])
print(shoot)
guess = shoot.pack_arcs(shoot.pack([-2.5, -0.18, 0.005], [-0.03], 0.2), [(0.03, 0.1)])
sol = shoot.solve(guess)
(t_in, t_out), = shoot.unpack_arcs(sol.v)
tr = shoot.trajectory(sol.v, n_points=800)
print(f"|R| = {sol.residual_norm:.1e}; singular arc on [{t_in:.6f}, {t_out:.6f}], "
      f"t_f = {sol.tf:.8f}")
h_opt = tr["z"][-1, 0]
print(f"h(t_f) = {h_opt:.9f}  (reference 1.01283692; naive {h_naive:.8f})")
print(f"climb above the surface: {100 * ((h_opt - 1) / (h_naive - 1) - 1):.1f}% more than naive")

Throttling back on the singular arc buys a few percent more climb from the
same propellant: modest, but free, and it is what the necessary conditions
single out. Now the checks. A converged residual says the junction conditions
hold; `diagnostics` checks what the residual cannot:

In [ ]:
d = shoot.diagnostics(sol.v)
lo, hi = d["singular_u"][0]
print(f"singular thrust from {lo[0]:.3f} to {hi[0]:.3f} (bounds 0 and {umax})")
print(f"Legendre-Clebsch on the arc: from {d['singular_legendre'][0][0]:.3g} "
      f"to {d['singular_legendre'][0][1]:.3g} (must be >= 0)")
print(f"held-bound sign violation: {d['held_sign_violation'][0]:.1e} (must be <= 0)")
print(f"singular control outside its bounds by: {d['singular_bound_violation'][0]:.1e}")

The singular thrust stays inside $[0, 3.5]$; Legendre–Clebsch is positive, so
the arc can be minimizing; and on the free arcs the switching function has
the sign of the bound held (negative while thrust is full, positive while
coasting), so the bang-bang law agrees with the structure everywhere.

In [ ]:
S = np.array([kg.switching(ti, zi, P)[0] for ti, zi in zip(tr["t"], tr["z"])])
fig, axes = plt.subplots(1, 3, figsize=(12, 3.3))
axes[0].plot(tr["t"], tr["u"][:, 0], color=ORANGE, lw=2)
axes[0].set_title("thrust u", fontsize=10, loc="left")
axes[1].plot(tr["t"], S, color=BLUE, lw=2)
axes[1].axhline(0, color="k", lw=0.8)
axes[1].set_title("switching function S", fontsize=10, loc="left")
axes[2].plot(tr["t"], tr["z"][:, 0], color=BLUE, lw=2)
axes[2].set_title("altitude h", fontsize=10, loc="left")
for a_ in axes:
    for tj in (t_in, t_out):
        a_.axvline(tj, color=GRAY, lw=0.8, ls=":")
    a_.set_xlabel("t")
    a_.grid(alpha=0.3)
fig.tight_layout()

Full thrust, then the singular thrust, which *rises* steadily (from about
2.1 to 3.1) as the rocket climbs into thinner air where drag costs less,
then a coast. $S$ is negative, then pinned at
zero, then positive.

**Practicalities.** `max_nfev` (default 200000) caps the right-hand side
evaluations per arc: a bad guess can send the singular dynamics into a
region where $b \to 0$ and $u_s$ blows up, and the cap makes that fail fast
instead of crawling. `from_smoothed` builds a guess from a smoothed problem's
control history when no rough guess is at hand.

## 4. Without telling it the structure

The structure above was given. `Homotopy` finds it: it smooths the thrust,
follows the smoothing down, and reads off where the smoothed thrust stays
strictly between its bounds for a long time, a candidate singular arc
(Lecture 15).

In [ ]:
from auto_optimal_control.homotopy import Homotopy

ra = Homotopy(goddard).solve(P, ([-6, -0.5, -0.1], [-1, 0, 0.1]), (0.15, 0.3))
print(ra.message)
print("structure found:", ra.arcs)
print(f"h(t_f) = {-ra.objective:.9f}, t_f = {ra.solution.tf:.9f}")

## Summary

- A singular arc is an interval where the switching function vanishes
  identically. Differentiating $S$ until the control appears gives the
  singular control; the order $q$ is half the number of derivatives needed.
- Order-1 arcs: $u_s = -a/b$ from $\ddot S = a + bu$, Legendre–Clebsch
  $-b \ge 0$, entry with $S = \dot S = 0$. Order 2 and up chatter (Fuller).
- `derive(problem, singular=True)` generates the arc (`singular_arcs`:
  `derivatives`, `law`, `legendre_clebsch`, `junction`) or the reason it
  cannot (`singular_unsupported`).
- `MultiArcShooting` takes `("singular", u, before, after)` and
  `("switch", u)` entries; controls with singular arcs hold their bounds on
  free arcs; `diagnostics` reports the singular control's range,
  Legendre–Clebsch and the held-bound signs.

## Exercises

1. For the singular regulator with $a = 1$, $T = 4$, solve with
   `MultiArcShooting` and the structure `("singular", "u", "lower", "upper")`.
   Check $\lambda(0) = a^2$ and the cost $2a^3/3$.
2. Halve the drag coefficient $D_0$ in Goddard's problem. Does the singular
   arc get longer or shorter? Explain with the trade-off between drag and
   gravity loss.
3. Fuller's problem, $\min\int x^2$ with $\ddot x = u$, $|u| \le 1$, has an
   order-2 singular arc. Run `derive(..., singular=True)` on it and read the
   reason it gives. Then solve the smoothed problem for decreasing
   $\varepsilon$ and watch the switches multiply.

## Further reading

- H. S. Tsien and R. C. Evans, "Optimum thrust programming for a sounding
  rocket", *Journal of the American Rocket Society* 21, 1951.
- H. J. Kelley, R. E. Kopp and H. G. Moyer, "Singular extremals", in
  *Topics in Optimization* (G. Leitmann, ed., 1967).
- H. Seywald and E. M. Cliff, "Goddard problem in presence of a dynamic
  pressure limit", *Journal of Guidance, Control, and Dynamics* 16, 1993.
- K. Graichen and N. Petit, "Solving the Goddard problem with thrust and
  dynamic pressure constraints using saturation functions", IFAC World
  Congress, 2008.
- D. A. Clary, *Rocket Man: Robert H. Goddard and the Birth of the Space Age*
  (2003).